# 阶段 3：机器学习基线模型

## 任务说明

基于阶段 2 的特征数据，训练两个传统机器学习基线模型，用于预测下一交易日 SPY 的涨跌方向。

- **标签构造**: target_direction = (下一交易日 return_1d > 0) 为 1，否则为 0
- **模型**: LogisticRegression、RandomForestClassifier
- **划分方式**: 按时间顺序前 80% 训练、后 20% 测试（不打乱）
- **评估指标**: accuracy、precision、recall、f1

注意事项：
- 本阶段仅做方向预测的基线模型训练与评估
- 不涉及交易回测、LSTM、强化学习或交易策略
- 预测结果不代表投资建议

In [ ]:
import pandas as pd
from pathlib import Path

# 探测项目根目录
cwd = Path.cwd()
if (cwd / "data").exists() and (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "未找到项目根目录，请从项目根目录或 notebooks 目录启动 Notebook。"
    )

TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

FEAT_PATH = PROJECT_ROOT / "data" / "features" / "SPY_features_2015_2025.csv"
if FEAT_PATH.exists():
    df_feat = pd.read_csv(FEAT_PATH, index_col=0, parse_dates=True)
    print(f'特征数据: {df_feat.shape[0]} 行, {df_feat.shape[1]} 列')
    display(df_feat.head())
else:
    print(f'文件不存在: {FEAT_PATH.resolve()}')
    print('请先运行: python run_stage2.py')


In [ ]:
# 构造标签并展示分布
df_label = df_feat.copy()
df_label["target_return_1d"] = df_label["return_1d"].shift(-1)
df_label["target_direction"] = (df_label["target_return_1d"] > 0).astype(int)
df_label = df_label.dropna(subset=["target_return_1d", "target_direction"])

print(f"有效样本: {len(df_label)} 行")
print(f"方向分布:")
up_pct = 100 * df_label['target_direction'].mean()
print(f"  上涨 (1): {df_label['target_direction'].sum()}  ({up_pct:.1f}%)")
print(f"  下跌 (0): {(df_label['target_direction']==0).sum()}  ({100-up_pct:.1f}%)")


In [ ]:
# 读取评估指标
METRICS_PATH = TABLES_DIR / "SPY_ml_baseline_metrics.csv"

if METRICS_PATH.exists():
    df_metrics = pd.read_csv(METRICS_PATH)
    print(f'评估指标 ({len(df_metrics)} 行):')
    display(df_metrics)
else:
    print(f'文件不存在: {METRICS_PATH.resolve()}')
    print('请先运行: python run_stage3.py')


In [ ]:
# 读取测试集预测结果
PRED_PATH = TABLES_DIR / "SPY_ml_test_predictions.csv"

if PRED_PATH.exists():
    df_pred = pd.read_csv(PRED_PATH, index_col=0, parse_dates=True)
    print(f'测试集预测结果: {df_pred.shape[0]} 行, {df_pred.shape[1]} 列')
    print(f'字段: {list(df_pred.columns)}')
    print(f'日期范围: {df_pred.index.min().strftime("%Y-%m-%d")} ~ {df_pred.index.max().strftime("%Y-%m-%d")}')
    print()
    display(df_pred.head(10))
else:
    print(f'文件不存在: {PRED_PATH.resolve()}')
    print('请先运行: python run_stage3.py')


In [ ]:
# 读取特征重要性
IMP_PATH = TABLES_DIR / "SPY_ml_feature_importance.csv"

if IMP_PATH.exists():
    df_imp = pd.read_csv(IMP_PATH)
    print(f'特征重要性: {len(df_imp)} 行')
    print()
    for model_name in df_imp['model'].unique():
        subset = df_imp[df_imp['model'] == model_name].sort_values(
            'importance', ascending=False
        )
        print(f'=== {model_name} ===')
        for _, row in subset.iterrows():
            bar = '#' * int(row['importance'] / subset['importance'].max() * 30)
            print(f"  {row['feature']:20s} | {row['importance']:10.6f} | {bar}")
        print()
else:
    print(f'文件不存在: {IMP_PATH.resolve()}')
    print('请先运行: python run_stage3.py')


## 阶段 3 小结

- 基于阶段 2 的特征数据，构造了下一交易日涨跌方向标签
- 按时间顺序划分训练集 (80%) 和测试集 (20%)，未打乱时间序列
- 训练了两个传统机器学习基线模型：LogisticRegression 和 RandomForestClassifier
- 评估指标已保存至 `outputs/tables/SPY_ml_baseline_metrics.csv`
- 测试集预测结果已保存至 `outputs/tables/SPY_ml_test_predictions.csv`
- 特征重要性已保存至 `outputs/tables/SPY_ml_feature_importance.csv`
- 训练好的模型已保存至 `outputs/models/` 目录 (.joblib 格式)
- 以上结果仅用于模型解释和后续报告，不构成交易策略或投资建议
- 本阶段仅完成方向分类的基线训练、评估与结果整理，不涉及回测、LSTM、强化学习